# Урок 17 — Автомобілі як Об'єкти
## Інкапсуляція · Поліморфізм · Дандер-методи

---

> *«Дані — це не система. Система — це дані + поведінка + контроль.»*

---

У попередньому уроці ми навчились будувати ієрархії класів через наслідування.  
Але справжня сила ООП — не у тому, **як клас побудований**,  
а у тому, **як він захищає свій стан та взаємодіє зі світом**.

> **Три головні ідеї уроку:**
> 1. **Інкапсуляція** — об'єкт захищає свій внутрішній стан від хаосу ззовні
> 2. **Поліморфізм** — один виклик, різна поведінка залежно від типу
> 3. **Дандер-методи** — як ваш клас інтегрується у Python Runtime

---

## Зміст

| # | Розділ |
|---|--------|
| 1 | Завантаження даних — «просто таблиця» |
| 2 | Перша спроба — дані без поведінки |
| 3 | Базовий клас `Car` |
| 4 | Інкапсуляція — захист інваріантів |
| 5 | Поліморфізм — різні двигуни, різна логіка |
| 6 | Дандер-методи — об'єкти з поведінкою |
| 7 | Клас `CarFleet` — збираємо систему |
| 8 | Реальні дані назад — аналітика через об'єкти |
| 9 | Типові помилки студентів |
| 10 | Мінізавдання |
| 11 | Питання для інтерв'ю |
| 12 | Висновок |

---

## Розділ 1 — Завантаження даних

Ми використовуємо датасет `mpg` із `seaborn`.  
Він містить реальні дані про автомобілі 1970–1982 років:

| Поле | Значення |
|------|----------|
| `mpg` | Miles per gallon — паливна ефективність |
| `cylinders` | Кількість циліндрів |
| `displacement` | Об'єм двигуна (куб. дюйми) |
| `horsepower` | Потужність (к.с.) |
| `weight` | Вага (фунти) |
| `acceleration` | Розгін 0–60 mph (секунди) |
| `model_year` | Рік моделі (70–82) |
| `origin` | Виробник: `usa`, `europe`, `japan` |
| `name` | Назва моделі |

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

df = sns.load_dataset("mpg")
df = df.dropna()

print(f"Розмір датасету: {df.shape}")
df.head()

---

## ❌ Проблема 1 — це просто таблиця

У нас є дані:
- `mpg` — скільки миль на галон
- `weight` — вага автомобіля
- `origin` — країна виробника

Але це **просто цифри** у рядках.

Ми не можемо:
- контролювати правильність даних
- додати логіку (дизель ≠ бензин ≠ електро)
- описати реальний автомобіль як сутність

> Це НЕ система. Це просто DataFrame.

---

## Розділ 2 — Перша спроба: формули без логіки

In [ ]:
# Наївний підхід — рахуємо ефективність прямо в DataFrame
df["efficiency"] = df["mpg"] / df["weight"] * 1000

df[["name", "mpg", "weight", "origin", "efficiency"]].head(8)

## ❌ Проблема 2 — формула є, логіки немає

Ми щось порахували. Але:

- будь-хто може змінити `mpg = -50` — ніхто не зупинить
- дизель ≠ бензин ≠ електро — одна формула для всіх
- немає валідації, немає поведінки, немає захисту

> Це просто функція над рядком таблиці, а не модель реального світу.

---

## 🧠 Рішення — створюємо об'єкти

Тепер автомобіль — це не рядок таблиці.  
Це **об'єкт з поведінкою**.

Але одразу виникає питання:  
як зробити так, щоб дані всередині об'єкта **неможливо було зіпсувати**?

---

## Розділ 3 — Базовий клас `Car`

In [ ]:
class Car:
    def __init__(self, name, mpg, weight, origin):
        self.name = name
        self._mpg = mpg          # protected — домовленість не чіпати напряму
        self.__weight = weight   # private — Python ховає під іншим ім'ям
        self.origin = origin

    def info(self):
        return f"{self.name} ({self.origin})"

In [ ]:
# Перевіряємо
car = Car("Ford Pinto", 18.0, 2601, "usa")
print(car.info())
print(car._mpg)        # доступно, але "не рекомендовано"

try:
    print(car.__weight)  # Python ховає це
except AttributeError as e:
    print(f"AttributeError: {e}")

# Name mangling — як насправді зберігається __weight
print(car._Car__weight)

### Що відбувається з `__weight`?

Python застосовує **name mangling** — автоматичне перейменування:

```
__weight  →  _Car__weight
```

Це не справжній «замок», але це **механізм**, а не просто домовленість.

| Префікс | Тип | Механізм | Аналогія |
|---------|-----|----------|----------|
| `name` | public | немає | відкрите вікно |
| `_name` | protected | соціальний контракт | зачинені двері без замка |
| `__name` | private | name mangling | двері з замком (ключ є, але схований) |

---

## ❌ Проблема 3 — ми можемо зламати об'єкт

In [ ]:
car = Car("Ford Pinto", 18.0, 2601, "usa")

# Нічого не зупиняє нас від цього:
car._mpg = -100
print(f"mpg після злому: {car._mpg}")  # -100 — логіка зламана

> ### 💥 Що відбудеться, якщо викликати `efficiency()` після злому?
>
> Ми щойно записали `-100` у `_mpg`.
> Подивимось, що поверне розрахунок ефективності.

In [ ]:
# Демонструємо катастрофу:
# базова формула: efficiency = mpg / 10
broken = Car("Ford Pinto", 18.0, 2601, "usa")

print("До злому:  efficiency =", broken._mpg / 10)

broken._mpg = -100  # "злам"

print("Після злому: efficiency =", broken._mpg / 10)
print()
print("Від'ємна ефективність — це не edge-case.")
print("Це відсутність контролю над станом об'єкта."


> Ми дали доступ до даних — і їх зіпсували.  
> Потрібен **контроль**.

---

## Розділ 4 — Інкапсуляція: захист інваріантів

**Інкапсуляція** — це здатність об'єкта захищати свій внутрішній стан.  
Ми не закриваємо дані «назавжди» — ми **контролюємо доступ через методи**.

> Інваріант — це умова, яка завжди має бути істинною.  
> Для `mpg`: значення > 0. Для `weight`: 500 ≤ weight ≤ 5000.

In [ ]:
class Car:
    def __init__(self, name, mpg, weight, origin):
        self.name = name
        self._mpg = mpg
        self.__weight = weight
        self.origin = origin

    # --- Setter для mpg ---
    def set_mpg(self, value):
        if value <= 0:
            print(f"❌ Invalid mpg: {value}. Значення має бути > 0")
            return
        self._mpg = value

    # --- Getter для weight (private) ---
    def get_weight(self):
        return self.__weight

    def info(self):
        return f"{self.name} ({self.origin})"

In [ ]:
car = Car("Ford Pinto", 18.0, 2601, "usa")

# Спроба зламати — тепер заблокована
car.set_mpg(-50)
print(f"mpg після спроби злому: {car._mpg}")  # залишається 18.0

# Правильний спосіб змінити
car.set_mpg(20.0)
print(f"mpg після правильного оновлення: {car._mpg}")

# Доступ до private через getter
print(f"Вага: {car.get_weight()} фунтів")

### 🔒 Інкапсуляція — підсумок

```
car._mpg = -100        ❌  зламали логіку
car.set_mpg(-100)      ✅  об'єкт захистив себе
```

Ми не використовуємо `@property` — тільки явні методи `get_*` / `set_*`.  
Це **явний контроль**: читач коду одразу бачить, де є захист.

> **Інкапсуляція** = не «схований атрибут», а **захищений інваріант**.

---

## ❌ Проблема 4 — всі машини однакові

Уявіть наївний підхід:

```python
def fuel_efficiency(car):
    return car._mpg / 10
```

Але:
- **Бензиновий** двигун: базова ефективність  
- **Дизельний**: економніший на 20%  
- **Електричний**: у 2.5 рази ефективніший

Одна функція — неправда для всіх трьох.

---

### ❌ Наївний підхід — "if-hell"

Перш ніж дивитись на правильне рішення, подивімось, що бувають без поліморфізму:

```python
def bad_efficiency(car):
    if type(car).__name__ == "DieselCar":
        return car._mpg * 1.2
    elif type(car).__name__ == "ElectricCar":
        return car._mpg * 2.5
    else:
        return car._mpg * 1.0
```

> Додаєш `HybridCar` — лізеш у цю функцію.  
> Додаєш `HydrogenCar` — знову лізеш.  
> Логіка розкидана по коду — **не живе у класах**.  
>
> Це не ООП. Це **if-hell**.

---

---

## Розділ 5 — Поліморфізм: різні двигуни, різна логіка

**Поліморфізм** — здатність одного виклику `car.efficiency()` запускати **різну логіку** залежно від типу об'єкта.

Базовий клас задає **інтерфейс** (назву методу).  
Підкласи надають **конкретну реалізацію**.

In [ ]:
class GasolineCar(Car):
    """Бензиновий — базова ефективність."""
    def efficiency(self):
        return round(self._mpg * 1.0, 2)


class DieselCar(Car):
    """Дизельний — на 20% економніший."""
    def efficiency(self):
        return round(self._mpg * 1.2, 2)


class ElectricCar(Car):
    """Електричний — у 2.5 рази ефективніший."""
    def efficiency(self):
        return round(self._mpg * 2.5, 2)

In [ ]:
cars = [
    GasolineCar("Ford Pinto",   18.0, 2601, "usa"),
    DieselCar(  "VW Rabbit",    30.0, 1925, "europe"),
    ElectricCar("Tesla Model S", 100.0, 4647, "usa"),
]

print(f"{'Модель':<20} {'Тип':<12} {'Ефективність'}")
print("-" * 48)
for car in cars:
    car_type = type(car).__name__
    print(f"{car.info():<20} {car_type:<12} {car.efficiency()}")

### 🔥 Поліморфізм — підсумок

```python
for car in cars:
    print(car.efficiency())   # один виклик → різна логіка
```

Python сам вирішує, яку версію `efficiency()` викликати — залежно від типу об'єкта.  
Ми не пишемо `if isinstance(car, DieselCar): ...`

> **Поліморфізм** — це коли логіка живе у класі, а не у зовнішній функції з `if`.

---

## ❌ Проблема 5 — об'єкти «некрасиві»

In [ ]:
# Що ми бачимо без дандерів?
car = GasolineCar("Ford Pinto", 18.0, 2601, "usa")

print(car)          # <__main__.GasolineCar object at 0x...>
print([car])        # [<__main__.GasolineCar object at 0x...>]
print(car == car)   # True (порівняння за адресою пам'яті, не за значенням)

---

## Розділ 6 — Дандер-методи: об'єкти з поведінкою

**Дандер-методи** (double underscore = `__method__`) — це **контракт з Python Runtime**.  
Вони визначають, як ваш об'єкт поводиться у вбудованих операціях:

| Метод | Викликається при... | Приклад |
|-------|--------------------|---------|
| `__str__` | `print(obj)`, `str(obj)` | Гарне відображення |
| `__repr__` | `repr(obj)`, REPL | Debug-відображення |
| `__eq__` | `obj1 == obj2` | Порівняння |
| `__lt__` | `obj1 < obj2`, `sorted()` | Сортування |
| `__len__` | `len(obj)` | Розмір |
| `__getitem__` | `obj[index]` | Індексація |

In [ ]:
class Car:
    def __init__(self, name, mpg, weight, origin):
        self.name = name
        self._mpg = mpg
        self.__weight = weight
        self.origin = origin

    # --- Setter / Getter ---
    def set_mpg(self, value):
        if value <= 0:
            print(f"❌ Invalid mpg: {value}")
            return
        self._mpg = value

    def get_weight(self):
        return self.__weight

    def info(self):
        return f"{self.name} ({self.origin})"

    # --- Дандер-методи ---
    def __str__(self):
        """Для користувача: print(car)"""
        return f"{self.name} — {self._mpg} mpg [{self.origin}]"

    def __repr__(self):
        """Для розробника: car у REPL або repr(car)"""
        return f"Car(name={self.name!r}, mpg={self._mpg}, origin={self.origin!r})"

    def __eq__(self, other):
        """Рівність за mpg (з перевіркою типу!)"""
        if not isinstance(other, Car):
            return NotImplemented
        return self._mpg == other._mpg

    def __lt__(self, other):
        """Менший = менш ефективний. Потрібен для sorted()."""
        if not isinstance(other, Car):
            return NotImplemented
        return self._mpg < other._mpg

    def __len__(self):
        """len(car) повертає вагу (семантичний вибір)."""
        return int(self.__weight)

In [ ]:
# Оновлюємо підкласи, щоб вони успадкували нові дандери
class GasolineCar(Car):
    def efficiency(self):
        return round(self._mpg * 1.0, 2)

class DieselCar(Car):
    def efficiency(self):
        return round(self._mpg * 1.2, 2)

class ElectricCar(Car):
    def efficiency(self):
        return round(self._mpg * 2.5, 2)

In [ ]:
cars = [
    GasolineCar("Ford Pinto",    18.0, 2601, "usa"),
    DieselCar(  "VW Rabbit",     30.0, 1925, "europe"),
    GasolineCar("Toyota Corolla",34.0, 2200, "japan"),
    ElectricCar("Tesla Model S", 100.0, 4647, "usa"),
]

# __str__ — гарний print
print("--- print(car) ---")
for car in cars:
    print(car)

print()

# __repr__ — debug view
print("--- repr(car) ---")
print(repr(cars[0]))

> ### 🤔 Чи може Python сортувати `Car`-об'єкти без нашої допомоги?

```python
sorted([Car(...), Car(...)])
# TypeError: '<' not supported between instances of 'Car' and 'Car'
```

> Python **не знає**, що таке `Car`.  
> Він не знає, яка машина "менша" — поки ти не навчиш його через `__lt__`.
>
> Щойно ми визначили `__lt__` — `sorted()`, `min()`, `max()` починають **розуміти** наші об'єкти.

In [ ]:
# __lt__ + sorted() — магія
print("--- sorted(cars) — від найменш до найбільш ефективного ---")
for car in sorted(cars):
    print(car)

print()

# __eq__
a = GasolineCar("Honda Civic", 30.0, 1800, "japan")
b = DieselCar(  "VW Rabbit",   30.0, 1925, "europe")
print(f"a == b (однаковий mpg, різний тип): {a == b}")

# __len__
print(f"len(cars[0]) — вага Ford Pinto: {len(cars[0])} фунтів")

### ✨ Дандер-методи — підсумок

Ми навчили об'єкти:
- **красиво друкуватись** → `__str__`, `__repr__`
- **порівнюватись** → `__eq__`
- **сортуватись** → `__lt__` (Python викликає його в `sorted()`)
- **мати розмір** → `__len__`

> Дандер-методи — це не магія. Це **контракт з Python Runtime**.

```python
sorted(cars)   # Python викликає car.__lt__(other) для кожної пари
print(car)     # Python викликає car.__str__()
len(car)       # Python викликає car.__len__()
```

> `__str__` vs `__repr__`:  
> якщо `__str__` немає — Python використає `__repr__` як запасний варіант.

---

## Розділ 6.1 — Callable об'єкти: машина як функція

До цього ми навчили об'єкти:
- гарно **друкуватись** (через `__str__`)
- **порівнюватись** (через `__eq__`, `__lt__`)
- **сортуватись** (через `sorted()`)
- **мати розмір** (через `__len__`)

Але що якщо ми хочемо **"запустити" машину**?  
Наприклад — порахувати витрати пального на задану дистанцію.

> Тоді об'єкт може поводитись як функція — через `__call__`.

```python
car(100)   # "запускаємо" машину на 100 км
```

> Ми не викликаємо **функцію**.  
> Ми викликаємо **об'єкт**.  
> Об'єкт став поведінкою.

In [ ]:
# --- Пояснення перед кодом ---
# mpg = miles per gallon (скільки миль машина проїжджає на 1 галон пального)
#
# Але в Європі ми використовуємо:
# 👉 літри на 100 км (L/100km)
#
# Формула переходу:
# L/100km = 235.2 / mpg
#
# Звідки 235.2?
# 1 mile = 1.609 км
# 1 gallon = 3.785 літра
#
# Комбінація дає:
# (100 км / 1.609) * 3.785 ≈ 235.2
#
# Тобто:
# 👉 це просто коефіцієнт конвертації між системами вимірювання

# ------------------------------------------------------------

class GasolineCar(Car):
    def efficiency(self):
        # Бензиновий автомобіль — базова ефективність
        return round(self._mpg * 1.0, 2)

    def __call__(self, distance_km):
        """
        Дозволяє викликати об'єкт як функцію:
        car(distance_km) -> скільки літрів пального потрібно

        Наприклад:
        car(100) -> витрати на 100 км
        """

        # 🔒 Захист від некоректного вводу
        if distance_km <= 0:
            print("❌ Distance must be positive")
            return None

        # 🔁 Конвертуємо mpg -> L/100km
        # Чим більший mpg → тим менше витрати
        l_per_100km = 235.2 / self._mpg

        # 📏 Рахуємо витрати для конкретної дистанції
        # Наприклад:
        # якщо 8 л / 100 км
        # то на 200 км буде 16 л
        fuel_used = l_per_100km * (distance_km / 100)

        return round(fuel_used, 2)


# ------------------------------------------------------------

class DieselCar(Car):
    def efficiency(self):
        # Дизель економніший (~20%)
        return round(self._mpg * 1.2, 2)

    def __call__(self, distance_km):
        """
        Те саме, що бензин, але:
        👉 дизель ефективніший → фактичний mpg більший
        """

        if distance_km <= 0:
            print("❌ Distance must be positive")
            return None

        # 🔥 Поліморфізм:
        # той самий метод, але інша логіка
        effective_mpg = self._mpg * 1.2

        # Менші витрати → більший mpg → менше L/100km
        l_per_100km = 235.2 / effective_mpg

        fuel_used = l_per_100km * (distance_km / 100)

        return round(fuel_used, 2)


# ------------------------------------------------------------

class ElectricCar(Car):
    def efficiency(self):
        # Умовно вважаємо, що EV "ефективніший" у 2.5 рази
        return round(self._mpg * 2.5, 2)

    def __call__(self, distance_km):
        """
        Електромобіль НЕ використовує літри.

        Тому ми змінюємо модель:
        👉 повертаємо енергію (кВт·год)

        Типове значення:
        ~15 кВт·год / 100 км
        """

        if distance_km <= 0:
            print("❌ Distance must be positive")
            return None

        # ⚡ Інша фізика:
        # тут немає mpg → зовсім інша система
        energy_used = 15.0 * (distance_km / 100)

        return round(energy_used, 2)

In [ ]:
# Демонстрація __call__
gasoline = GasolineCar("Toyota Corolla", 34.0, 2200, "japan")
diesel   = DieselCar(  "VW Rabbit",      30.0, 1925, "europe")
electric = ElectricCar("Tesla Model S",  100.0, 4647, "usa")

print(f"{'Модель':<22} {'100 км':<12} {'250 км':<12} {'500 км':<12} {'Одиниці'}")
print("-" * 68)

for car in [gasoline, diesel, electric]:
    unit = "кВт·год" if isinstance(car, ElectricCar) else "л"
    print(f"{car.info():<22} {str(car(100)):<12} {str(car(250)):<12} {str(car(500)):<12} {unit}")

> ### 🔥 Поліморфізм + `__call__`
>
> Кожна машина **по-різному відповідає** на один виклик `car(distance)`:
>
> - `GasolineCar(100)` → літри бензину
> - `DieselCar(100)` → літри дизеля (менше!)
> - `ElectricCar(100)` → кВт·год
>
> ```python
> for car in cars:
>     print(car(100))   # Python викликає car.__call__(100)
> ```
>
> Один виклик — різна логіка — **поліморфізм + callable** разом.

---

## Розділ 7 — Клас `CarFleet`: збираємо систему

Один автомобіль — це об'єкт.  
Парк автомобілів — це **система**.  

Ми будуємо `CarFleet` — клас, що керує колекцією машин і сам поводиться як колекція.

In [ ]:
class CarFleet:
    def __init__(self, cars):
        self.cars = list(cars)

    def __len__(self):
        """len(fleet) → кількість машин у парку"""
        return len(self.cars)

    def __getitem__(self, index):
        """fleet[i] → пряма індексація"""
        return self.cars[index]

    def __repr__(self):
        return f"CarFleet({len(self)} cars)"

    def most_efficient(self):
        """Повертає найефективнішу машину (за mpg)."""
        return max(self.cars)

    def sorted_by_efficiency(self):
        """Повертає список, відсортований від найменш до найбільш ефективного."""
        return sorted(self.cars)

In [ ]:
fleet = CarFleet(cars)

print(f"Розмір парку: {len(fleet)} машини")
print(f"Перша машина: {fleet[0]}")
print(f"Остання машина: {fleet[-1]}")
print()
print(f"Найефективніша: {fleet.most_efficient()}")
print()
print("Всі машини за ефективністю:")
for car in fleet.sorted_by_efficiency():
    print(f"  {car}")

> ### 💡 Це вже не список
>
> `CarFleet` не просто обгортає `list`.  
> Це **інтерфейс до системи**.
>
> ```python
> len(fleet)                # Python викликає fleet.__len__()
> fleet[0]                  # Python викликає fleet.__getitem__(0)
> fleet.most_efficient()    # бізнес-логіка через систему
> ```
>
> Зовнішній код не знає, як саме зберігаються машини всередині.  
> Він лише використовує **поведінку**.

---

## Розділ 8 — Реальні дані: аналітика через об'єкти

Повертаємось до DataFrame — але тепер будуємо об'єкти зі справжніх рядків.

In [ ]:
# Створюємо список Car з реального датасету
real_cars = [
    GasolineCar(row["name"], row["mpg"], row["weight"], row["origin"])
    for _, row in df.iterrows()
    if pd.notna(row["mpg"]) and pd.notna(row["weight"])
]

real_fleet = CarFleet(real_cars)
print(f"Реальний парк: {len(real_fleet)} автомобілів")
print(f"Найефективніший: {real_fleet.most_efficient()}")
print()
print("Топ-5 за ефективністю:")
for car in real_fleet.sorted_by_efficiency()[-5:]:
    print(f"  {car}")

In [ ]:
# Візуалізація — середній mpg по країні походження
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

origins = {}
for car in real_fleet:
    origins.setdefault(car.origin, []).append(car._mpg)

avg_mpg = {k: sum(v)/len(v) for k, v in origins.items()}
sorted_origins = sorted(avg_mpg, key=avg_mpg.get)

axes[0].bar(sorted_origins, [avg_mpg[o] for o in sorted_origins], color="steelblue")
axes[0].set_title("Середній MPG по країні")
axes[0].set_ylabel("MPG")

efficiencies = [car.efficiency() for car in real_fleet]
axes[1].hist(efficiencies, bins=20, color="salmon", edgecolor="black")
axes[1].set_title("Розподіл ефективності")
axes[1].set_xlabel("Efficiency")

plt.tight_layout()
plt.show()

---

## Розділ 9 — Типові помилки студентів

> Ці помилки — **не просто баги**. Вони показують де саме ламається мислення.

In [ ]:
# ❌ Помилка 1 — лізуть напряму в _mpg
car = GasolineCar("Ford", 20.0, 2000, "usa")
car._mpg = -100  # "зламали"
print(f"Ефективність після злому: {car.efficiency()}")
print()
print("✅ Правильно: car.set_mpg(-100)  →  перевірка всередині класу")

In [ ]:
# ❌ Помилка 2 — думають що __weight доступний напряму
car = GasolineCar("Ford", 20.0, 2000, "usa")
try:
    print(car.__weight)
except AttributeError as e:
    print(f"AttributeError: {e}")

# Name mangling
print(f"✅ Правильно: car._Car__weight = {car._Car__weight}")
print("або: car.get_weight()")

In [ ]:
# ❌ Помилка 3 — if-hell замість поліморфізму
def bad_efficiency(car):
    """❌ Це не ООП"""
    if isinstance(car, DieselCar):
        return car._mpg * 1.2
    elif isinstance(car, ElectricCar):
        return car._mpg * 2.5
    else:
        return car._mpg * 1.0

# ✅ Правильно
def good_efficiency(car):
    """✅ Поліморфізм — логіка у класі"""
    return car.efficiency()

test_car = DieselCar("VW", 30.0, 1925, "europe")
print(f"bad:  {bad_efficiency(test_car)}")
print(f"good: {good_efficiency(test_car)}")

In [ ]:
# ❌ Помилка 4 — __eq__ без перевірки типу
class BadCar:
    def __init__(self, mpg):
        self._mpg = mpg
    def __eq__(self, other):
        return self._mpg == other._mpg  # падає якщо other не має _mpg

bc = BadCar(30.0)
try:
    print(bc == "not a car")
except AttributeError as e:
    print(f"AttributeError: {e}")

print()
print("✅ Правильно: перевіряти isinstance або повертати NotImplemented")

In [ ]:
# ❌ Помилка 5 — sorted() без __lt__
class CarWithoutLt:
    def __init__(self, name, mpg):
        self.name = name
        self._mpg = mpg

cars_bad = [CarWithoutLt("A", 20.0), CarWithoutLt("B", 30.0)]
try:
    sorted(cars_bad)
except TypeError as e:
    print(f"TypeError: {e}")

print()
print("✅ Правильно: визначити __lt__ → sorted() починає розуміти об'єкти")

---

## Розділ 10 — Мінізавдання

### 🔒 Завдання 1 — Encapsulation

Додай до класу `Car` методи `set_weight()` і `get_mpg()`:
- `set_weight(value)` — валідує: `500 ≤ weight ≤ 5000`
- `get_mpg()` — повертає `_mpg`

In [ ]:
# TODO: Завдання 1
# Розширте клас Car методами set_weight() і get_mpg()

class Car:
    def __init__(self, name, mpg, weight, origin):
        self.name   = name
        self._mpg   = mpg
        self.__weight = weight
        self.origin = origin

    def set_mpg(self, value):
        if value <= 0:
            print(f"❌ Invalid mpg: {value}")
            return
        self._mpg = value

    def get_weight(self):
        return self.__weight

    # TODO: додати set_weight() і get_mpg()

    def info(self):
        return f"{self.name} ({self.origin})"

### 🔥 Завдання 2 — Polymorphism

Додай клас `HybridCar`:
- `efficiency = mpg * 1.7`
- `__call__(distance_km)` → витрати пального (використай формулу GasolineCar, але з ефективним mpg)

In [ ]:
# TODO: Завдання 2
# class HybridCar(Car):
#     def efficiency(self):
#         return round(self._mpg * 1.7, 2)
#
#     def __call__(self, distance_km):
#         ...

# hybrid = HybridCar("Toyota Prius", 50.0, 2800, "japan")
# print(hybrid.efficiency())
# print(hybrid(100))

### ✨ Завдання 3 — Dunder

Додай до `Car`:
1. `__gt__` — більше ніж
2. `__len__` — вже є, але поясни чому вага = довжина

Потім:
```python
print(max(cars))
print(len(cars[0]))
```

In [ ]:
# TODO: Завдання 3
# Додай __gt__ до Car, потім:
# print(max(cars))
# print(len(cars[0]))

### 🧠 Завдання 4 — Fleet

Розшир клас `CarFleet`:
- Додай метод `get_heaviest_car()` — повертає найважчу машину
- Додай метод `average_efficiency()` — середня ефективність парку
- Додай `__iter__` — щоб `for car in fleet` працювало напряму

In [ ]:
# TODO: Завдання 4
# class CarFleet:
#     ...
#     def get_heaviest_car(self):
#         ...
#     def average_efficiency(self):
#         ...
#     def __iter__(self):
#         ...

### 📊 Завдання 5 — Реальні дані

Використай `df` (mpg dataset):
1. Створи список `real_cars` тільки з японських машин
2. Відсортуй за ефективністю
3. Виведи топ-3

```python
# japan_cars = [...]
# sorted_cars = sorted(japan_cars)
# for car in sorted_cars[-3:]:
#     print(car)
```

In [ ]:
# TODO: Завдання 5
# japan_cars = [
#     GasolineCar(row["name"], row["mpg"], row["weight"], row["origin"])
#     for _, row in df.iterrows()
#     if row["origin"] == "japan" and pd.notna(row["mpg"])
# ]
# sorted_cars = sorted(japan_cars)
# for car in sorted_cars[-3:]:
#     print(car)

---

## Розділ 11 — Питання для інтерв'ю

> Це не просто домашка — це підготовка до реального технічного інтерв'ю.

**Питання 1:** Чим відрізняється `_name` від `__name` в Python?

**Питання 2:** Як реалізувати поліморфізм без `isinstance`?

**Питання 3:** Що робить `sorted()` якщо об'єкт не реалізує `__lt__`?

**Питання 4:** Для чого `__repr__` якщо є `__str__`?

**Питання 5:** Що таке callable об'єкт? Як перевірити, чи є об'єкт callable?

```python
car = GasolineCar("Toyota", 34.0, 2200, "japan")
print(callable(car))   # True — бо визначений __call__
```

**Питання 6:** Чому `__eq__` повинен повертати `NotImplemented` замість `False` для невідомих типів?

---

## Розділ 12 — Висновок

### Що ми збудували

| Концепт | Реалізація | Результат |
|---------|------------|----------|
| **Інкапсуляція** | `_mpg`, `__weight`, `set_mpg()`, `get_weight()` | Об'єкт захищає свій стан |
| **Поліморфізм** | `GasolineCar`, `DieselCar`, `ElectricCar` — кожен зі своїм `efficiency()` | Один виклик, різна поведінка |
| **Дандер-методи** | `__str__`, `__repr__`, `__eq__`, `__lt__`, `__len__` | Об'єкт інтегрований у Python Runtime |
| **Callable** | `__call__` — об'єкт як функція | `car(100)` рахує витрати |
| **Система** | `CarFleet` з `__len__`, `__getitem__` | Колекція, що сама поводиться як список |

---

### Еволюція підходу

```
DataFrame → рядки з числами
         ↓
Car      → об'єкт з інваріантами
         ↓
GasolineCar / DieselCar / ElectricCar → поліморфізм
         ↓
car(100) → callable об'єкт
         ↓
CarFleet → система з поведінкою
```

---

### Що змінилось у нашому мисленні

```
Початок: DataFrame

  дані відкриті
  логіка відсутня
  система неконтрольована

Кінець: система об'єктів

  об'єкти захищають себе (інкапсуляція)
  поведінка визначена у класах (поліморфізм)
  Python розуміє наші об'єкти (дандер-методи)
  об'єкт можна "викликати" як функцію (__call__)
```

> **Інженерне мислення** — це не просто "написати код".  
> Це побудова системи, де дані **захищені**, поведінка **передбачувана**,  
> а Python **розуміє** ваші об'єкти нативно.
>
> ООП — це не класи.  
> ООП — це **контроль реальності через код**.